# Taller 10 - MongoDB




### Para empezar a utilizar mongodb desde 0, vamos a partir realizando los siguientes pasos:


1.   Instalaremos $\texttt{mongodb}$
2.   Inicializaremos el proceso $\texttt{mongodb}$
3.   Instalaremos el comando $\texttt{gdown}$ para bajar la base de datos en formato json de google drive.
4.   Importaremos los datos por linea de comando. Dejaremos los datos en la base de datos $\texttt{local}$, en la colección $\texttt{pokedex}$, y le informaremos al comando que los datos vienen como una arreglo de jsons.

In [ ]:
%%bash
set -euo pipefail

rm -f /etc/apt/sources.list.d/mongodb-org-6.0.list

apt-get update
apt-get install -y curl gnupg ca-certificates

# Configurar el repositorio compatible con Ubuntu 24.04.
curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc \
  | gpg --batch --yes --dearmor \
      -o /usr/share/keyrings/mongodb-server-8.0.gpg

echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-8.0.gpg ] https://repo.mongodb.org/apt/ubuntu noble/mongodb-org/8.0 multiverse" \
  > /etc/apt/sources.list.d/mongodb-org-8.0.list

apt-get update
apt-get install -y mongodb-org

# Iniciar MongoDB dentro de Colab.
mkdir -p /data/db
if ! pgrep -x mongod > /dev/null; then
  mongod --dbpath /data/db \
    --bind_ip 127.0.0.1 \
    --fork --logpath /tmp/mongodb.log
fi

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://repo.mongodb.org/apt/ubuntu noble/mongodb-org/8.0 InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
curl is already the newest version (8.5.0-2ubuntu10.16).
gnupg is already the newest version (2.4.4-2ubuntu17.6).
ca-certificates is already the newest version (20260816ubuntu0.24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu nobl

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
!gdown https://drive.google.com/uc?id=108aLE4mFE5bZYitFfX5c14Qj_aYA5MO_
!mongoimport --db local --collection pokedex --drop --jsonArray < pokedex.json

Downloading...
From: https://drive.google.com/uc?id=108aLE4mFE5bZYitFfX5c14Qj_aYA5MO_
To: /content/pokedex.json
100% 2.95M/2.95M [00:00<00:00, 34.8MB/s]
2026-10-27T18:36:26.849+0000	connected to: mongodb://localhost/
2026-10-27T18:36:26.849+0000	dropping: `local.pokedex`
2026-10-27T18:36:27.495+0000	809 document(s) imported successfully. 0 document(s) failed to import.


A continuación importaremos la librería de mongo para python, crearemos un cliente que se conectara a $\texttt{localhost}$ por defecto, nos conectaremos a la base local, y luego a la colección pokedex la cual almacenaremos en la variable $\texttt{collection}$.

In [ ]:
pip install pymongo

In [ ]:
from pymongo import MongoClient
client = MongoClient()
client.list_database_names() # ['admin', 'local']
db = client['local']
collection = db['pokedex']

Para partir, podemos listar toda la colección usando el comando $\texttt{collection.find({})}$. Para no saturar la salida, nos limitaremos a mirar el primero de los registros.

In [ ]:
import pprint
for pokemon in collection.find({}).limit(1):
    pprint.pprint(pokemon)

{'Abilities': ['Overgrow', 'Chlorophyll'],
 'Attack': 49,
 'Capture Rate': 45,
 'Category': 'Seed Pokémon',
 'Defense': 49,
 'Egg Steps': 5120,
 'Exp Group': 'Medium Slow',
 'HP': 45,
 'Height (ft)': '2\'04"',
 'Height (m)': 0.7,
 'Id': '001',
 'Moves': {'Double-Edge': {'Accuracy': 100,
                           'Description': 'A reckless, life-risking tackle. '
                                          'This also damages the user quite a '
                                          'lot.',
                           'Effect %': '--',
                           'Level': 27,
                           'PP': 15,
                           'Power': 120,
                           'Type': 'Normal'},
           'Growl': {'Accuracy': 100,
                     'Description': 'The user growls in an endearing way, '
                                    'making opposing Pokémon less wary. This '
                                    'lowers their Attack stat.',
                     'Effect %': '--'

Fijense que aparte de los datos básicos del pokemon, también se presentan las habilidades (una lista de strings), y los movimientos (un arreglo asociativo/diccionario de los movimientos y sus detalles).

Si queremos ahora listar los pokemones que se llaman $\texttt{Charmander}$ podemos hacer la siguiente consulta:

In [ ]:
for pokemon in collection.find({'Name': 'Charmander'}):
    pprint.pprint(pokemon)

{'Abilities': ['Blaze', 'Solar Power'],
 'Attack': 52,
 'Capture Rate': 45,
 'Category': 'Lizard Pokémon',
 'Defense': 43,
 'Egg Steps': 5120,
 'Exp Group': 'Medium Slow',
 'HP': 39,
 'Height (ft)': '2\'00"',
 'Height (m)': 0.6,
 'Id': '004',
 'Moves': {'Dragon Rage': {'Accuracy': 100,
                           'Description': 'This attack hits the target with a '
                                          'shock wave of pure rage. This '
                                          'attack always inflicts 40 HP '
                                          'damage.',
                           'Effect %': '--',
                           'Level': 16,
                           'PP': 10,
                           'Power': '??',
                           'Type': 'Dragon'},
           'Ember': {'Accuracy': 100,
                     'Description': 'The target is attacked with small flames. '
                                    'This may also leave the target with a '
                         

Y si queremos proyectar algunos de sus atributos lo hacemos tal como si fuera un diccionario:

In [ ]:
for pokemon in collection.find({'Name': 'Mew'}):
    print(pokemon['Speed'],pokemon['Abilities'])

100 ['Synchronize']


Recomendamos hechar un vistazo [aquí](https://docs.mongodb.com/manual/tutorial/query-documents/) para ver otros ejemplos de consultas que utilizarán en las siguientes preguntas:

#Pregunta 1
Encuentre los nombres de los pokémon que miden más de 3.0 m.
[Ordene](https://www.w3schools.com/python/python_mongodb_sort.asp) por altura descendente.

In [ ]:
for p in collection.find({'Height (m)': {'$gt': 3.0}}).sort('Height (m)', -1):
    print(p['Name'], "- Altura:", p['Height (m)'])

Wailord - Altura: 14.5
Celesteela - Altura: 9.2
Steelix - Altura: 9.2
Onix - Altura: 8.8
Rayquaza - Altura: 7
Gyarados - Altura: 6.5
Milotic - Altura: 6.2
Yveltal - Altura: 5.8
Stakataka - Altura: 5.5
Guzzlord - Altura: 5.5
Dialga - Altura: 5.4
Lugia - Altura: 5.2
Zygarde - Altura: 5
Kyogre - Altura: 4.5
Giratina - Altura: 4.5
Palkia - Altura: 4.2
Dragonair - Altura: 4
Lunala - Altura: 4
Dhelmise - Altura: 3.9
Xurkitree - Altura: 3.8
Ho-Oh - Altura: 3.8
Regigigas - Altura: 3.7
Naganadel - Altura: 3.6
Arbok - Altura: 3.5
Groudon - Altura: 3.5
Solgaleo - Altura: 3.4
Serperior - Altura: 3.3
Reshiram - Altura: 3.2
Arceus - Altura: 3.2


#Pregunta 2
Encuentre los nombres de los pokemones ''Fragile Speedster'', que son los que tienen velocidad sobre 100 y defensa bajo 60.

In [ ]:
for p in collection.find({'Speed': {'$gt': 100}, 'Defense': {'$lt': 60}}):
    print(p['Name'])

Raichu
Dugtrio
Alakazam
Kadabra
Electabuzz
Sneasel
Ninjask
Deoxys
Floatzel
Liepard
Swoobat
Cryogonal
Accelgor
Heliolisk
Dedenne
Nihilego
Pheromosa
Blacephalon


#Pregunta 3

Liste los nombres y speed de los pokemones cuyo tipo principal (Type 1) sea "Fire". Muestre únicamente esos dos campoos, sin incluir _id.

In [ ]:
for p in collection.find({'Type 1': 'Fire'}, {'Name': 1, 'Speed': 1, '_id': 0}):
    print(p['Name'], "-", p['Speed'])

Charmander - 65
Charmeleon - 80
Charizard - 100
Ninetales - 100
Vulpix - 65
Growlithe - 60
Arcanine - 95
Ponyta - 90
Rapidash - 105
Magmar - 93
Flareon - 65
Moltres - 90
Cyndaquil - 65
Quilava - 80
Typhlosion - 100
Slugma - 20
Magcargo - 30
Magby - 83
Entei - 100
Ho-Oh - 90
Torchic - 45
Combusken - 55
Blaziken - 80
Numel - 35
Camerupt - 40
Torkoal - 20
Chimchar - 61
Monferno - 81
Infernape - 108
Magmortar - 83
Heatran - 77
Pignite - 55
Tepig - 45
Emboar - 65
Pansear - 64
Simisear - 101
Darumaka - 50
Darmanitan - 95
Heatmor - 65
Fennekin - 60
Braixen - 73
Delphox - 104
Talonflame - 126
Fletchinder - 84
Litleo - 72
Pyroar - 106
Volcanion - 70
Litten - 70
Torracat - 90
Incineroar - 60
Oricorio - 93
Turtonator - 36
Blacephalon - 107


#Pregunta 4
Seleccione los pokémon cuya Category contiene la palabra “Dragon”. Usa el operador [regex](https://www.mongodb.com/docs/manual/reference/operator/query/regex/).

In [ ]:
for p in collection.find({'Category': {'$regex': r'\bDragon\b'}}):
    print(p['Name'], ":", p['Category'])

Horsea : Dragon Pokémon
Seadra : Dragon Pokémon
Dratini : Dragon Pokémon
Dragonair : Dragon Pokémon
Dragonite : Dragon Pokémon
Kingdra : Dragon Pokémon
Salamence : Dragon Pokémon
Goodra : Dragon Pokémon


#Pregunta 5
Seleccione los pokémon que tienen el movimiento Double-Edge y cuya Accuracy es 100.

**Hint**: El acceso a campos anidados se hace con [“dot notation”](https://www.mongodb.com/docs/manual/core/document/#dot-notation).

In [ ]:
for p in collection.find({'Moves.Double-Edge.Accuracy': 100}):
    print(p['Name'])

Bulbasaur
Ivysaur
Venusaur
Rattata
Raticate
Jigglypuff
Wigglytuff
Geodude
Graveler
Golem
Onix
Cubone
Marowak
Chansey
Eevee
Cyndaquil
Quilava
Ledyba
Typhlosion
Ledian
Togepi
Togetic
Marill
Sudowoodo
Azumarill
Sunkern
Sunflora
Pineco
Dunsparce
Forretress
Steelix
Phanpy
Blissey
Linoone
Skitty
Aron
Lairon
Aggron
Volbeat
Numel
Spinda
Chimecho
Relicanth
Bagon
Shelgon
Salamence
Kyogre
Jirachi
Bonsly
Hippopotas
Hippowdon
Victini
Audino
Venipede
Scolipede
Whirlipede
Basculin
Deerling
Sawsbuck
Escavalier
Karrablast
Larvesta
Skiddo
Gogoat
Bergmite
Avalugg
Wishiwashi
Stufful
Bewear
Passimian
Silvally
Type: Null
Minior
Celesteela
Stakataka


#Pregunta 6

Liste los pokémon monotipo: aquellos que no tienen Type 2 (no existe o es "None").

**Hint**: Puede testear si es que una llave existe utilizando [$exists](https://docs.mongodb.com/manual/reference/operator/query/exists/)


In [ ]:
for p in collection.find({'$or': [{'Type 2': 'None'}, {'Type 2': {'$exists': False}}]}):
    print(p['Name'])

Charmander
Charmeleon
Squirtle
Blastoise
Caterpie
Metapod
Wartortle
Rattata
Raticate
Arbok
Pikachu
Ekans
Sandshrew
Sandslash
Raichu
Nidorina
Nidoran♀
Nidorino
Clefairy
Nidoran♂
Clefable
Ninetales
Vulpix
Diglett
Dugtrio
Meowth
Persian
Psyduck
Mankey
Primeape
Growlithe
Arcanine
Poliwag
Poliwhirl
Abra
Golduck
Alakazam
Machop
Kadabra
Machamp
Machoke
Ponyta
Rapidash
Seel
Grimer
Muk
Shellder
Drowzee
Hypno
Krabby
Voltorb
Electrode
Kingler
Cubone
Marowak
Hitmonlee
Hitmonchan
Koffing
Weezing
Chansey
Lickitung
Tangela
Kangaskhan
Horsea
Seadra
Goldeen
Staryu
Seaking
Electabuzz
Magmar
Pinsir
Magikarp
Ditto
Tauros
Vaporeon
Jolteon
Flareon
Eevee
Porygon
Snorlax
Dratini
Dragonair
Mewtwo
Mew
Chikorita
Bayleef
Meganium
Cyndaquil
Quilava
Totodile
Croconaw
Feraligatr
Sentret
Furret
Typhlosion
Pichu
Cleffa
Togepi
Flaaffy
Ampharos
Bellossom
Mareep
Sudowoodo
Politoed
Aipom
Sunkern
Espeon
Sunflora
Umbreon
Unown
Wobbuffet
Misdreavus
Pineco
Dunsparce
Granbull
Snubbull
Ursaring
Slugma
Teddiursa
Remoraid
Phanpy


#Pregunta 7
Seleccione los pokémon que tienen [alguna](https://www.mongodb.com/docs/manual/reference/operator/query/in/) de las siguientes habilidades: Levitate o Intimidate.

In [ ]:
for p in collection.find({'Abilities': {'$in': ['Levitate', 'Intimidate']}}):
    print(p['Name'])

Arbok
Ekans
Growlithe
Arcanine
Gastly
Haunter
Koffing
Weezing
Gyarados
Tauros
Unown
Misdreavus
Granbull
Qwilfish
Snubbull
Stantler
Hitmontop
Mightyena
Masquerain
Mawile
Vibrava
Flygon
Lunatone
Solrock
Claydol
Duskull
Chimecho
Baltoy
Salamence
Latias
Latios
Staraptor
Staravia
Shinx
Luxray
Luxio
Mismagius
Chingling
Bronzor
Bronzong
Carnivine
Uxie
Mesprit
Rotom
Giratina
Cresselia
Azelf
Herdier
Stoutland
Sandile
Krokorok
Krookodile
Scraggy
Scrafty
Eelektrik
Eelektross
Tynamo
Cryogonal
Hydreigon
Landorus
Litten
Torracat
Incineroar
Vikavolt


#Pregunta 8
Liste los pokémon que tienen [exactamente](https://www.mongodb.com/docs/manual/reference/operator/query/size/) 4 habilidades.

In [ ]:
for p in collection.find({'Abilities': {'$size': 4}}):
    print(p['Name'], "- Habilidades:", p['Abilities'])

Basculin - Habilidades: ['Reckless', 'Rock Head', 'Adaptability', 'Mold Breaker']
Meowstic - Habilidades: ['Keen Eye', 'Infiltrator', 'Prankster', 'Competitive']
Rockruff - Habilidades: ['Keen Eye', 'Vital Spirit', 'Steadfast', 'Own Tempo']


#Pregunta 9

Considere la primera generación de pokemon (IDs entre "001" y "151"). Liste los pokémon con: "001" ≤ Id ≤ "151", Capture Rate entre 45 y 90, Type 2 exista y sea ≠ "None" y cuyo Nombre empiece con M o S.

**Hint**: Los siguientes operadores les sera de ayuda:

1. [Comparación](https://www.mongodb.com/docs/manual/reference/operator/query-comparison/)

2. [Regex](https://www.mongodb.com/docs/manual/reference/operator/query/regex/)


In [ ]:
for p in collection.find(
    {
        'Id': {'$gte': '001', '$lte': '151'},
        'Capture Rate': {'$gte': 45, '$lte': 90},
        'Type 2': {'$ne': 'None', '$exists': True},
        'Name': {'$regex': r'^(M|S)'}
    },
    {'Name': 1, 'Id': 1, 'Capture Rate': 1, 'Type 2': 1, '_id': 0}
):
    print(p['Id'], p['Name'])


080 Slowbro
082 Magneton
122 Mr. Mime
123 Scyther
121 Starmie


#Pregunta 10

Cree un [índice](https://www.mongodb.com/docs/manual/reference/method/db.collection.createIndex/) sobre Category y devuelva nombres y categorías de los pokémon cuya categoría contenga “Mouse” o “Seed”, excluyendo los que contengan “Cotton”.

In [ ]:
collection.create_index([('Category', 'text')])

'Category_text'

In [ ]:
for p in collection.find({'$text': {'$search': 'Mouse Seed -Cotton'}}, {'Name': 1, 'Category': 1, '_id': 0}):
    print(p['Name'], ":", p['Category'])

Raichu : Mouse Pokémon
Sandslash : Mouse Pokémon
Sandshrew : Mouse Pokémon
Pikachu : Mouse Pokémon
Raticate : Mouse Pokémon
Rattata : Mouse Pokémon
Bidoof : Plump Mouse Pokémon
Pichu : Tiny Mouse Pokémon
Cyndaquil : Fire Mouse Pokémon
Sunkern : Seed Pokémon
Venusaur : Seed Pokémon
Ivysaur : Seed Pokémon
Bulbasaur : Seed Pokémon
Ferroseed : Thorn Seed Pokémon


#Pregunta 11 (PROPUESTO)

Encuentre los pokemones que cumplan las siguientes condiciones:

1. Speed > 100.

2. Defensa < 80.

3. Tiene al menos la habilidad "Levitate" o "Intimidate".

Para cada pokemón muestre el nombre, velocidad, defensa y habilidades, excluyendo el id. Además, ordene los resultados por velocidad descendente, con criterio de desempate el nombre en ascendente.

In [ ]:
consulta = {
    'Speed': {'$gt': 100},
    'Defense': {'$lt': 80},
    'Abilities': {'$in': ['Levitate', 'Intimidate']}
}

proyeccion = {
    'Name': 1,
    'Speed': 1,
    'Defense': 1,
    'Abilities': 1,
    '_id': 0
}

for p in collection.find(consulta, proyeccion).sort([('Speed', -1), ('Name', 1)]):
    print(p)

{'Name': 'Azelf', 'Abilities': ['Levitate'], 'Defense': 70, 'Speed': 115}
{'Name': 'Cryogonal', 'Abilities': ['Levitate'], 'Defense': 50, 'Speed': 105}
{'Name': 'Mismagius', 'Abilities': ['Levitate'], 'Defense': 60, 'Speed': 105}


#Pregunta 12 (PROPUESTO)
Encuentre los 5 pokemones con mayor ataque que tengan Attack > 90, que además tengan un movimiento que cumpla las siguientes condiciones al mismo tiempo:

1. Su nombre sea "Double-Edge" o "Body Slam"
2. Su precisión es igual a 100.
3. Su potencia es mayor o igual a 80.

Por cada pokemon, muestre únicamente el nombre y su ataque, sin incluir el id. Además, ordene el ataque de manera descendente, y en caso de empate, por nombre ascendente

In [ ]:
consulta = {
    'Attack': {'$gt': 90},
    '$or': [
        {
            'Moves.Double-Edge.Accuracy': 100,
            'Moves.Double-Edge.Power': {'$gte': 80}
        },
        {
            'Moves.Body Slam.Accuracy': 100,
            'Moves.Body Slam.Power': {'$gte': 80}
        }
    ]
}

proyeccion = {
    'Name': 1,
    'Attack': 1,
    '_id': 0
}

for p in collection.find(consulta, proyeccion).sort([('Attack', -1), ('Name', 1)]).limit(5):
    print(p)

{'Name': 'Escavalier', 'Attack': 135}
{'Name': 'Salamence', 'Attack': 135}
{'Name': 'Stakataka', 'Attack': 131}
{'Name': 'Bewear', 'Attack': 125}
{'Name': 'Pangoro', 'Attack': 124}
